##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# -----------------------------
# 1) Load CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# Keep labels as integers (SparseCategoricalCrossentropy)
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")



170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


In [2]:
# -----------------------------
# 2) Data augmentation
# -----------------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")


In [3]:
# -----------------------------
# 3) Build MobileNetV2 backbone (pretrained)
# -----------------------------
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
mobilenet_base.trainable = False  # freeze first (feature extractor)



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [4]:
# -----------------------------
# 4) Full model (preprocess inside model)
# -----------------------------
mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),     # IMPORTANT: correct for MobileNetV2
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(10)                      # logits
], name="cifar10_mobilenetv2")



In [5]:
# -----------------------------
# 5) Inspect architecture
# -----------------------------
mobilenet_model.summary()

# Extra prints for the lab requirements
print("\n--- Architecture Observations ---")
print("Network depth (number of layers):", len(mobilenet_model.layers))
print("Total params:", mobilenet_model.count_params())
print("Trainable params:", int(np.sum([tf.size(w).numpy() for w in mobilenet_model.trainable_weights])))
print("Non-trainable params:", int(np.sum([tf.size(w).numpy() for w in mobilenet_model.non_trainable_weights])))

print("\nBackbone layers trainable (frozen stage):",
      sum(l.trainable for l in mobilenet_base.layers), "/", len(mobilenet_base.layers))



Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


--- Architecture Observations ---
Network depth (number of layers): 7
Total params: 2270794
Trainable params: 12810
Non-trainable params: 2257984

Backbone layers trainable (frozen stage): 0 / 154


In [6]:
# -----------------------------
# 6) Compile + Train (frozen backbone)
# -----------------------------
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
]

history = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)



Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 82s 104ms/step - accuracy: 0.5665 - loss: 1.2394 - val_accuracy: 0.8106 - val_loss: 0.5312 - learning_rate: 0.0010
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 75s 107ms/step - accuracy: 0.7122 - loss: 0.8184 - val_accuracy: 0.8224 - val_loss: 0.5105 - learning_rate: 0.0010
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 75s 106ms/step - accuracy: 0.7323 - loss: 0.7702 - val_accuracy: 0.8086 - val_loss: 0.5502 - learning_rate: 0.0010


In [7]:
# -----------------------------
# 7) Test / Evaluate (frozen)
# -----------------------------
test_loss_frozen, test_acc_frozen = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("\nMobileNetV2 (frozen) test accuracy:", test_acc_frozen)
print("MobileNetV2 (frozen) test loss    :", test_loss_frozen)



MobileNetV2 (frozen) test accuracy: 0.8144000172615051
MobileNetV2 (frozen) test loss    : 0.532081663608551


In [8]:
# -----------------------------
# 8) Fine-tune last layers
# -----------------------------
mobilenet_base.trainable = True

# Freeze all but last 30 layers in the backbone 
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

print("\nBackbone layers trainable (fine-tune stage):",
      sum(l.trainable for l in mobilenet_base.layers), "/", len(mobilenet_base.layers))

# Re-compile with a small LR
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)




Backbone layers trainable (fine-tune stage): 30 / 154
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 110s 142ms/step - accuracy: 0.6379 - loss: 1.0693 - val_accuracy: 0.8330 - val_loss: 0.4882 - learning_rate: 1.0000e-05
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.7538 - loss: 0.7109 - val_accuracy: 0.8408 - val_loss: 0.4494 - learning_rate: 1.0000e-05
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.7768 - loss: 0.6378 - val_accuracy: 0.8504 - val_loss: 0.4228 - learning_rate: 1.0000e-05


In [9]:
# -----------------------------
# 9) Test / Evaluate (fine-tuned)
# -----------------------------
test_loss_ft, test_acc_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("\nMobileNetV2 (fine-tuned) test accuracy:", test_acc_ft)
print("MobileNetV2 (fine-tuned) test loss    :", test_loss_ft)




MobileNetV2 (fine-tuned) test accuracy: 0.8490999937057495
MobileNetV2 (fine-tuned) test loss    : 0.44037383794784546


In [10]:
# -----------------------------
# 10) Collect results
# -----------------------------
results = {
    "MobileNet frozen test acc": float(test_acc_frozen),
    "MobileNet fine-tuned test acc": float(test_acc_ft),
}
print("\n--- Results Summary ---")
for k, v in results.items():
    print(f"{k}: {v}")


--- Results Summary ---
MobileNet frozen test acc: 0.8144000172615051
MobileNet fine-tuned test acc: 0.8490999937057495


In [ ]:
### Questions:

#- Which model achieved the highest accuracy? MobileNetv2 achived an accuracy of 81.44% for frozen feature extractor, and 84.9% for fine-tuned model. On the other hand, the ResNet50 model achieved an accuracy of 88% for frozen feature extractor, and 91.9% for fine-tuned model. So, the ResNet50 model achieved the highest accuracy in both frozen and fine-tuned stages.
#- Which model trained faster? ResNet50v2 took around 9 minutes to train, while it took around 12 minutes for fine tune. For MobileNetv2 it took 4 minutes to train frozen feature extractor, and around 5.30 minutes for fine-tuning. So, MobileNet trained faster than ResNet50 in both stages.
#- How might the architecture explain the differences? ResNet50v2 is deeper and uses residual connections, which allow it to learn more complex features and achieve higher accuracy. In contrast, MobileNetV2 uses depthwise separable convolutions to reduce computations, making it faster and more efficient but with slightly lower accuracy.